In [1]:
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [18]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/ghmm_results/"

In [19]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01']

In [20]:
RUNS = 2
SPLITS = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]].keys())
num_states = 18
TARGET_LENGTHS=[9]

# 2. Parse ghmm models

In [21]:
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from hmm_logic_methods import save_model
from pomegranate import *

def parse_model(xml_root):
    hmm_tag = xml_root[0]
    Pi = []
    B = []
    Ends = []
        ## Parse Pi - initial probabilities
    for child in hmm_tag:
        if child.tag == "state":
            Pi.append(float(child.attrib["initial"]))
            ## Parse B - emission probs for each state
            emission_list = child[0].text.split(", ")
            emission_list = list(map(float,emission_list))
            B.append(emission_list[:-1])
            
            
        if child.tag == "alphabet":
            alphabet = list()
            for symbol_tag in child:
                if symbol_tag.text != 'Z':
                    alphabet.append(symbol_tag.text)
    end_state_index = len(B) - 1
    # print("End state index ", end_state_index)
    # Does not count end state
    Pi = Pi[:-1]
    B = B[:-1]
    
                             
    num_states = len(Pi)
    A = np.zeros(shape=(num_states,num_states))
        ## Parse A - transition probabilities
    for child in hmm_tag:
        if child.tag == "transition":
            source = int(child.attrib["source"])
            target = int(child.attrib["target"])
            probability = float(child[0].text)
    #        print(f" from {source} to  {target} with prob {probability}")
            if target != end_state_index:
                A[source, target] = probability
            else:
                # print("From",source,"To" ,"->" ,target, probability)
                Ends.append(probability)
    return Pi, A, B, Ends, np.unique(alphabet)




for allele_name in ALLELES:
    for run_num in range(RUNS):
        for split_num in range(SPLITS):
            target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
            name_prefix="{}_run".format(run_num)
            lengths_string = "-".join(map(str, TARGET_LENGTHS))
            name_postfix = name_prefix + "-" + lengths_string
            params_string = "ghmm_{}-st".format(num_states)
            model_name = "{}-{}_model-{}-{}".format(name_postfix,  params_string, target_allele_name, split_num)
            tree = ET.parse(f'{TARGET_PATH_TO_RESULTS}/{model_name}/{model_name}.xml')
            root = tree.getroot()
            Pi, A, B, Ends, alphabet = parse_model(root)
            
            distributions = list()
            for emission_probs in B:
                distribution = DiscreteDistribution({key: prob for key, prob in zip(alphabet,emission_probs)})
                distributions.append(distribution)
            state_names = ["s{:03d}".format(i) for i in range(len(Pi))]
            model = HiddenMarkovModel.from_matrix(name=model_name, transition_probabilities=A, 
                                      starts=Pi, distributions=distributions, ends=Ends, state_names=state_names, verbose=True)
            save_model(TARGET_PATH_TO_RESULTS, model)
            

In [22]:
model_name

'1_run-9-ghmm_18-st_model-HLA_A_02_01-4'